<br/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="left"/>
<img src="images/LOGO-CORREO_x.png" alt="SkillNest" width="280px" align="right"/>
<div align="center">
<h2>Bootcamp Data Science — Modulo 2</h2><br/>
<h1>Semana 8 · Miercoles — XGBoost</h1>
<h3>El padre del boosting moderno, todavia rey en muchos casos</h3>
<br/>
    <b>Instructor:</b> Jesus Ortiz · jesus.jeduardo7@gmail.com<br/><br/>
    <b>SkillNest · b2b-sonda-data-science</b>
</div>
<br/>

## Objetivos

Al final de la clase van a poder:

1. Entender que diferencia a XGBoost de LightGBM (level-wise vs leaf-wise).
2. Dominar los hiperparametros clave de XGBoost.
3. Combinar XGBoost con Optuna para tuning profesional.
4. Decidir cuando usar XGBoost vs LightGBM segun el contexto.
5. Resolver UN ejercicio tipo examen sobre prediccion de churn en una telco real.

# 1. XGBoost: el clasico

XGBoost (eXtreme Gradient Boosting) salio en 2014. Durante anos fue el modelo dominante en competencias Kaggle. Sigue siendo extremadamente competitivo y muchas empresas lo usan en produccion porque es robusto, esta bien documentado y tiene una comunidad enorme.

LightGBM (2017) lo superó en velocidad pero NO siempre en accuracy. En datasets medianos (< 100k filas) XGBoost suele ganar de poquito en metricas, especialmente cuando se tunea bien.

| | XGBoost | LightGBM |
|---|---|---|
| Crecimiento del arbol | Level-wise (nivel por nivel) | Leaf-wise (la mejor hoja) |
| Velocidad | Mas lento | 5-20x mas rapido |
| Robustez al overfitting | Mas robusto | Mas propenso si num_leaves es alto |
| Manejo de nulos | Aprende automaticamente | Aprende automaticamente |
| Manejo de categoricas | Necesita encoding | Nativo |
| Sweet spot | Datasets medianos, accuracy critica | Datasets grandes, velocidad importa |

Instalacion: `pip install xgboost`

# 2. Level-wise vs Leaf-wise: imagen mental

**XGBoost crece por niveles**: en cada paso, divide TODAS las hojas del nivel actual. El arbol queda balanceado.

```
        Nivel 0:     [raiz]
                       /  \
        Nivel 1:    [A]   [B]      <- divide ambas
                    / \   / \
        Nivel 2: [C][D][E][F]
```

**LightGBM crece por hoja**: en cada paso, elige UNA hoja para dividir (la que mas reduce el error). El arbol queda desbalanceado.

```
        [raiz]
         /  \
        [A]  [B]
              /\
            [C][D]
                /\
              [E][F]      <- arbol mucho mas profundo en una rama
```

Consecuencias:
- LightGBM converge mas rapido en cada iteracion (encuentra mejoras mas grandes).
- XGBoost es mas estable y menos propenso a sobre-ajustar.
- En datasets chicos, XGBoost suele ser mejor opcion porque LightGBM puede crecer ramas que sobreajustan.

# 3. Hiperparametros clave de XGBoost

Son MUY parecidos a LightGBM. Los nombres cambian un poco:

| XGBoost | LightGBM | Que hace |
|---|---|---|
| `n_estimators` | `n_estimators` | Numero de arboles |
| `learning_rate` (eta) | `learning_rate` | Cuanto contribuye cada arbol |
| `max_depth` | `max_depth` | Profundidad maxima |
| `min_child_weight` | `min_child_samples` | Peso minimo en hoja |
| `subsample` | `subsample` | % filas por arbol |
| `colsample_bytree` | `colsample_bytree` | % features por arbol |
| `gamma` | (no tiene equivalente directo) | Min ganancia para split (poda) |
| `reg_alpha` | `reg_alpha` | Regularizacion L1 |
| `reg_lambda` | `reg_lambda` | Regularizacion L2 |

XGBoost no tiene `num_leaves` porque crece level-wise. La profundidad la controlan con `max_depth` directamente.

## Setup

Hoy usamos un dataset NUEVO y muy comun en la industria: **Telco Customer Churn**. Es de IBM, contiene 7,043 clientes de una telco con 21 variables sobre su contrato, servicios y demograficos, y el target es si el cliente se va (churn) o se queda.

Es un caso de negocio realista: las telcos pierden mucha plata cuando un cliente cancela. Detectarlo temprano para intervenir vale literalmente millones.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix, classification_report)
import lightgbm as lgb
import xgboost as xgb

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

# Telco Customer Churn (IBM). Carga rapido desde GitHub.
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(URL)

# Limpieza minima
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

# Target binario
y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])

# Categoricas a codes (XGBoost necesita numericas, LightGBM las maneja igual)
cat_cols = X.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category').cat.codes

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Tasa de churn (% se va): {y.mean():.3f}')
print(f'Distribucion test: {y_test.value_counts().to_dict()}')

# 4. XGBoost basico

Primer entrenamiento con defaults. Para clasificacion binaria, `XGBClassifier` con `eval_metric='logloss'`.

In [ ]:
modelo_xgb = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

t0 = time.time()
modelo_xgb.fit(X_train, y_train)
t_xgb = time.time() - t0

pred = modelo_xgb.predict(X_test)
proba = modelo_xgb.predict_proba(X_test)[:, 1]

print(f'Accuracy:        {accuracy_score(y_test, pred):.4f}')
print(f'F1:              {f1_score(y_test, pred):.4f}')
print(f'Recall (churn=1): {recall_score(y_test, pred):.4f}')
print(f'AUC-ROC:         {roc_auc_score(y_test, proba):.4f}')
print(f'Tiempo:          {t_xgb:.2f}s')

# 5. Comparacion practica XGBoost vs LightGBM

Mismo dataset, mismos hiperparametros razonables, mismas semillas. Vamos a ver quien gana en este caso.

In [ ]:
modelo_lgb = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)

t0 = time.time(); modelo_lgb.fit(X_train, y_train); t_lgb = time.time() - t0
pred_lgb = modelo_lgb.predict(X_test); proba_lgb = modelo_lgb.predict_proba(X_test)[:, 1]

tabla = pd.DataFrame({
    'Modelo': ['XGBoost', 'LightGBM'],
    'Accuracy':  [accuracy_score(y_test, pred), accuracy_score(y_test, pred_lgb)],
    'F1':        [f1_score(y_test, pred), f1_score(y_test, pred_lgb)],
    'Recall churn': [recall_score(y_test, pred), recall_score(y_test, pred_lgb)],
    'AUC':       [roc_auc_score(y_test, proba), roc_auc_score(y_test, proba_lgb)],
    'Tiempo (s)': [round(t_xgb, 2), round(t_lgb, 2)]
}).set_index('Modelo').round(4)
print(tabla)

# 6. Tuning de XGBoost con Optuna

Mismo workflow que con LightGBM pero con los hiperparametros propios de XGBoost.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0, 10)
    }
    modelo = xgb.XGBClassifier(**params, random_state=42, eval_metric='logloss')
    return cross_val_score(modelo, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)

print(f'Mejor AUC CV: {study.best_value:.4f}')
print(f'Mejores params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

modelo_t = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss').fit(X_train, y_train)
pred_t = modelo_t.predict(X_test); proba_t = modelo_t.predict_proba(X_test)[:, 1]
print(f'\nEn test: AUC={roc_auc_score(y_test, proba_t):.4f}  F1={f1_score(y_test, pred_t):.4f}  Recall={recall_score(y_test, pred_t):.4f}')

---

# EJERCICIO TIPO EXAMEN — Customer Churn Telco

**Tiempo: 90 minutos · Entrega individual · 100 puntos**

> Trabajan como Data Scientists en una telco. El area comercial les dice: "estamos perdiendo clientes y no sabemos a cuales atacar con campanas de retencion. Tenemos presupuesto para llamar a 500 clientes este mes. Necesitamos que predigan quien tiene mas probabilidad de irse, para llamarlos primero."

Este ejercicio se evalua con la **rubrica al final**. NO es un ejercicio de codigo, es un ejercicio de **decision tecnica con justificacion**. El codigo es solo el medio, lo que importa es la calidad del analisis.

## Reglas del juego

- Trabajan con el mismo dataset Telco Churn que vimos en la demo.
- Pueden usar cualquier modelo, pero al menos uno tiene que ser XGBoost.
- NO pueden modificar el split de train/test (random_state=42, test_size=0.2, stratify=y).
- Toda decision tecnica debe estar **justificada con numeros o argumento de negocio**.
- Si copian y pegan analisis sin entender, se nota en la rubrica.

## Lo que tienen que entregar

### Parte 1 — EDA con foco en churn (15 pts)

1. Mostrar la distribucion del target. Cual es la tasa de churn? Hay desbalance?
2. Encontrar y graficar las **3 features que mas correlacionan con churn**. Justifiquen por que estas y no otras.
3. Una pregunta de analisis: hay alguna variable demografica (gender, SeniorCitizen, Partner, Dependents) que prediga churn? Que les dice esto sobre el negocio?

### Parte 2 — Baselines (15 pts)

4. Entrenar 3 modelos baseline: LogisticRegression (con StandardScaler en Pipeline), RandomForestClassifier, XGBClassifier.
5. Reportar tabla con Accuracy, Precision, Recall (sobre churn=1), F1 y AUC-ROC.
6. Cual modelo gana en AUC? Cual gana en Recall? Son los mismos? Por que?

### Parte 3 — XGBoost tuneado con Optuna (25 pts)

7. Definir un objective de Optuna con los 9 hiperparametros principales. Justificar la metrica que optimizan (no copien la de clase, **piensen** en el negocio).
8. Correr 30 trials.
9. Reportar mejores hiperparametros y metricas en test.
10. Comparar las 4 versiones: 3 baselines + XGBoost tuneado. Quien gano? Justifiquen con la matriz de confusion del ganador.

### Parte 4 — Decision de negocio (25 pts)

El equipo comercial tiene presupuesto para llamar a **500 clientes** este mes. Ustedes tienen que decidir cuales.

11. Tomen las probabilidades de churn que predice su mejor modelo sobre el TEST set. Ordenenlas de mayor a menor. Seleccionen los TOP 500.
12. De esos 500 clientes que ustedes recomiendan llamar, **cuantos realmente eran churners**? (lift sobre random)
13. Si llamaran al azar a 500 del test, cuantos churners agarrarian en promedio? (calculen)
14. Cual es la mejora multiplicativa que su modelo aporta sobre random? Esto es lo que se llama **lift**. Si es 3x, su modelo es 3 veces mejor que llamar al azar.
15. Si retener un cliente vale $500 y una llamada cuesta $20, calculen el ROI de su campana: cuanta plata genera y cuanta plata cuesta.

### Parte 5 — Defensa final (20 pts)

Escriban 1 pagina (en celda markdown) defendiendo su modelo ante el VP Comercial. Debe responder:

16. Por que XGBoost (o el ganador) y no otra cosa?
17. Cual es el riesgo del modelo? Que pasa cuando se equivoca?
18. Donde el modelo es debil? Para que tipo de clientes prediria mal?
19. Que harian si tuvieran 3 meses mas para mejorarlo? (sean concretos: nuevas features, A/B testing, etc.)
20. Hay algun sesgo demografico que les preocupe? Mirando la Parte 1.3, hay alguna variable que NO deberian usar por etica/legal (ej. genero)?

## Rubrica de evaluacion

| Parte | Que evaluo | Puntos |
|---|---|---|
| 1 | EDA: variables correctas + analisis demografico tiene sentido | 15 |
| 2 | Baselines bien entrenados + comparacion correcta entre AUC y Recall | 15 |
| 3 | Optuna corre + metrica justificada + comparacion con matriz de confusion | 25 |
| 4 | Calculo de lift y ROI correctos + interpretacion de negocio | 25 |
| 5 | Defensa logica + reconoce sesgos + propone mejoras concretas | 20 |
| **Total** | | **100** |

## Bonus (hasta 15 pts extra)

- (+5) Combinar Optuna con early stopping en el objective.
- (+5) Aplicar `scale_pos_weight` o `is_unbalanced` para manejar desbalance: cambia el ranking de la Parte 4?
- (+5) Calcular el ROI optimo: cual seria el N optimo de clientes a llamar (no necesariamente 500)?

In [ ]:
# Parte 1 — EDA con foco en churn



In [ ]:
# Parte 2 — Baselines



In [ ]:
# Parte 3 — XGBoost tuneado con Optuna



In [ ]:
# Parte 4 — Decision de negocio (lift y ROI)



# Parte 5 — Defensa final

(Escribir 1 pagina justificando el modelo elegido)


## Cierre

Hoy aprendimos:

- XGBoost crece level-wise (mas estable), LightGBM crece leaf-wise (mas agresivo).
- En datasets medianos XGBoost suele dar mejores resultados, en datasets grandes LightGBM gana en velocidad.
- El tuning de XGBoost con Optuna usa los mismos principios que con LightGBM.
- Lo mas importante de un modelo de churn NO es la accuracy: es el **lift** (cuanto mejor que random) y el **ROI** de la campana de retencion.

Con esto cerramos la semana 8. El proximo lunes empezamos con aprendizaje no supervisado y los proyectos finales.